<a href="https://colab.research.google.com/github/hayahanyyy/Bachelor-Thesis/blob/main/Datasett2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Install required libraries
!pip install pandas numpy scikit-learn xgboost catboost matplotlib seaborn -q

# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

Importing libraries

In [4]:
df = pd.read_csv('AI_Personalized_Learning.csv')

Loading dataset

In [5]:
df.head()

,student_id,age,gender,education_level,learning_style,previous_gpa,completed_modules,avg_time_per_module,engagement_score,distraction_events,quiz_accuracy,feedback_score,contextual_difficulty_level,recommended_path,actual_path_followed,path_efficiency_score,final_assessment_score,learning_outcome
0,STU001,20,Male,UG,Visual,3.75,5,42.70,67,4,100,4,Medium,M1→M2→M4,M1→M3→M5,84,69,Excellent
1,STU002,24,Female,UG,Kinesthetic,2.55,3,31.29,65,2,79,2,Medium,M1→M3→M5,M1→M3→M5,83,68,Fail
2,STU003,22,Male,PG,Visual,3.51,6,52.92,84,5,70,1,Medium,M1→M3→M5,M1→M2→M4,77,84,Good
3,STU004,23,Female,High School,Auditory,3.52,4,40.10,79,0,65,3,Medium,M1→M3→M5,M1→M2→M4,78,91,Fair
4,STU005,21,Female,High School,Kinesthetic,3.19,10,35.34,72,3,50,5,Hard,M1→M2→M4,M1→M3→M5,75,68,Fair


Reading dataset

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   student_id                   1000 non-null   object 
 1   age                          1000 non-null   int64  
 2   gender                       1000 non-null   object 
 3   education_level              1000 non-null   object 
 4   learning_style               1000 non-null   object 
 5   previous_gpa                 1000 non-null   float64
 6   completed_modules            1000 non-null   int64  
 7   avg_time_per_module          1000 non-null   float64
 8   engagement_score             1000 non-null   int64  
 9   distraction_events           1000 non-null   int64  
 10  quiz_accuracy                1000 non-null   int64  
 11  feedback_score               1000 non-null   int64  
 12  contextual_difficulty_level  1000 non-null   object 
 13  recommended_path   

**Data Preparation**

In [7]:
df.isnull().sum()

,0
student_id,0
age,0
gender,0
education_level,0
learning_style,0
previous_gpa,0
completed_modules,0
avg_time_per_module,0
engagement_score,0
distraction_events,0


Checking missing values



*   No missing values



In [8]:
df.duplicated().sum()

np.int64(0)

Checking for duplicates


*   No duplicate values




In [9]:
df = df.drop('student_id', axis=1)

Dropped student_id column because it doesn't help in prediction

In [10]:
df.head()

,age,gender,education_level,learning_style,previous_gpa,completed_modules,avg_time_per_module,engagement_score,distraction_events,quiz_accuracy,feedback_score,contextual_difficulty_level,recommended_path,actual_path_followed,path_efficiency_score,final_assessment_score,learning_outcome
0,20,Male,UG,Visual,3.75,5,42.70,67,4,100,4,Medium,M1→M2→M4,M1→M3→M5,84,69,Excellent
1,24,Female,UG,Kinesthetic,2.55,3,31.29,65,2,79,2,Medium,M1→M3→M5,M1→M3→M5,83,68,Fail
2,22,Male,PG,Visual,3.51,6,52.92,84,5,70,1,Medium,M1→M3→M5,M1→M2→M4,77,84,Good
3,23,Female,High School,Auditory,3.52,4,40.10,79,0,65,3,Medium,M1→M3→M5,M1→M2→M4,78,91,Fair
4,21,Female,High School,Kinesthetic,3.19,10,35.34,72,3,50,5,Hard,M1→M2→M4,M1→M3→M5,75,68,Fair


**Data Preprocessing**

In [11]:
cols_to_map = [
    'gender', 'education_level', 'learning_style',
    'contextual_difficulty_level', 'recommended_path',
    'actual_path_followed', 'learning_outcome'
]

# 3. Apply mapping using Category Codes
for col in cols_to_map:
    df[col] = df[col].astype('category')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   age                          1000 non-null   int64   
 1   gender                       1000 non-null   category
 2   education_level              1000 non-null   category
 3   learning_style               1000 non-null   category
 4   previous_gpa                 1000 non-null   float64 
 5   completed_modules            1000 non-null   int64   
 6   avg_time_per_module          1000 non-null   float64 
 7   engagement_score             1000 non-null   int64   
 8   distraction_events           1000 non-null   int64   
 9   quiz_accuracy                1000 non-null   int64   
 10  feedback_score               1000 non-null   int64   
 11  contextual_difficulty_level  1000 non-null   category
 12  recommended_path             1000 non-null   category
 13  actu

Changing all categorical columns to type category

Set target value 'y' to learning_outcome while 'x' all other features

In [20]:
# List of categorical features and target
categorical_cols = ['gender', 'education_level', 'learning_style',
                    'contextual_difficulty_level', 'recommended_path',
                    'actual_path_followed']
target_col = 'learning_outcome'

print("--- Categorical Mappings ---")
for col in categorical_cols + [target_col]:
    # A. Change type to 'category'
    df[col] = df[col].astype('category')

    # B. Print the value to code mapping
    # .cat.categories provides the names in the order they will be encoded
    print(f"\nColumn: {col}")
    for code, category in enumerate(df[col].cat.categories):
        print(f"  {category} -> {code}")

    # C. Do the mappings (Apply the codes)
    df[col] = df[col].cat.codes

# Define X and y AFTER encoding the categorical columns in df
X = df.drop('learning_outcome', axis=1)
y = df['learning_outcome']

--- Categorical Mappings ---

Column: gender
  0 -> 0
  1 -> 1

Column: education_level
  0 -> 0
  1 -> 1
  2 -> 2

Column: learning_style
  0 -> 0
  1 -> 1
  2 -> 2

Column: contextual_difficulty_level
  0 -> 0
  1 -> 1
  2 -> 2

Column: recommended_path
  0 -> 0
  1 -> 1

Column: actual_path_followed
  0 -> 0
  1 -> 1

Column: learning_outcome
  0 -> 0
  1 -> 1
  2 -> 2
  3 -> 3


Encoded all categorical columns into numerical

**Data Modelling **

In [23]:
# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Split the data (80% train, 20% test)

In [24]:
from sklearn.preprocessing import StandardScaler

# 1. Identify numerical columns (excluding your encoded categoricals)
numerical_cols = [
    'age', 'previous_gpa', 'completed_modules', 'avg_time_per_module',
    'engagement_score', 'distraction_events', 'quiz_accuracy',
    'feedback_score', 'path_efficiency_score', 'final_assessment_score'
]

# 2. Initialize the Scaler
scaler = StandardScaler()

# 3. Fit and Transform only the numerical features
# (Note: Do NOT scale your categorical codes or target variable)
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Numerical features have been standardized (Mean=0, Std=1).")

Numerical features have been standardized (Mean=0, Std=1).


Scaling features. Only scales numerical columns. Useful when using SVM and Logistic Regression.
Better done before splitting.

In [25]:
# MODEL INITIALIZATION
# ============================================================================

print("\n\n[STEP 3] INITIALIZING CLASSIFICATION MODELS\n")

# Define all 5 models
models = {
    'Logistic Regression (LR)': LogisticRegression(
        max_iter=1000,
        random_state=42,
        multi_class='multinomial',
        solver='lbfgs'
    ),
    'Support Vector Machine (SVM)': SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        random_state=42,
        probability=True
    ),
    'Random Forest (RF)': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_depth=None,
        min_samples_split=2
    ),
    'XGBoost (XGB)': XGBClassifier(
        n_estimators=100,
        random_state=42,
        verbosity=0,
        max_depth=6,
        learning_rate=0.1
    ),
    'CatBoost (CB)': CatBoostClassifier(
        iterations=100,
        random_state=42,
        verbose=0,
        depth=6,
        learning_rate=0.1
    )
}

print("Models initialized:")
for model_name, model in models.items():
    print(f"  ✓ {model_name}")



[STEP 3] INITIALIZING CLASSIFICATION MODELS

Models initialized:
  ✓ Logistic Regression (LR)
  ✓ Support Vector Machine (SVM)
  ✓ Random Forest (RF)
  ✓ XGBoost (XGB)
  ✓ CatBoost (CB)


Initializing all models

In [27]:
# MODEL TRAINING WITH 10-FOLD CROSS-VALIDATION

print("\n\n[STEP 4] TRAINING MODELS WITH 10-FOLD CROSS-VALIDATION\n")

# Dictionary to store results
cv_results = {}
trained_models = {}

# Scoring metrics
scoring_metrics = {
    'accuracy': 'accuracy',
    'precision_weighted': 'precision_weighted',
    'recall_weighted': 'recall_weighted',
    'f1_weighted': 'f1_weighted'
}

# Define StratifiedKFold for cross-validation
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Train each model
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    print("-" * 80)

    # Perform cross-validation
    cv_scores = cross_validate(
        model,
        X,
        y,
        cv=skf,
        scoring=scoring_metrics,
        return_train_score=True,
        n_jobs=-1
    )

    cv_results[model_name] = cv_scores



[STEP 4] TRAINING MODELS WITH 10-FOLD CROSS-VALIDATION


Training Logistic Regression (LR)...
--------------------------------------------------------------------------------

Training Support Vector Machine (SVM)...
--------------------------------------------------------------------------------

Training Random Forest (RF)...
--------------------------------------------------------------------------------

Training XGBoost (XGB)...
--------------------------------------------------------------------------------

Training CatBoost (CB)...
--------------------------------------------------------------------------------
